<a href="https://colab.research.google.com/github/EstherWare/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/Copy_of_colab_complete.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Duka Sales and Agent Cleanup

This self-contained notebook regenerates the source data, completes both engagements, validates the outputs, and writes the required CSV and Excel deliverables.


In [ ]:
import numpy as np
import pandas as pd

# Generator 1: Duka Electronics (seed 2026)
rng_duka = np.random.default_rng(2026)
n_duka = 1000
products_duka = {
    "Laptop": 105000,
    "Phone": 40000,
    "Tablet": 28000,
    "Headphones": 8500,
    "Smartwatch": 18000,
}
duka_sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n_duka),
    "order_date": pd.date_range("2026-01-01", "2026-06-30", periods=n_duka),
    "branch": rng_duka.choice(["Nairobi", "Mombasa", "Nakuru", "Kisumu", "Eldoret"], n_duka),
    "product": rng_duka.choice(list(products_duka.keys()), n_duka),
    "quantity": rng_duka.integers(1, 6, n_duka),
    "channel": rng_duka.choice(["In-store", "Online", "Phone order"], n_duka, p=[0.55, 0.35, 0.10]),
})
duka_sales["unit_price"] = duka_sales["product"].map(products_duka)
duka_sales.to_csv("duka_sales.csv", index=False)

# Generator 2: Agent Network (seed 44)
rng = np.random.default_rng(44)
n = 600
agents = pd.DataFrame({
    "agent_id": [f"AG{i:03d}" for i in range(1, 21)],
    "county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),
    "onboarded": pd.date_range("2024-01-01", periods=20, freq="MS").strftime("%Y-%m-%d"),
})
txns = pd.DataFrame({
    "txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],
    "agent_id": rng.choice(agents["agent_id"], n),
    "txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ", "Withdrawal", "deposit"], n),
    "amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
    "txn_date": (pd.to_datetime("2026-03-01") + pd.to_timedelta(rng.integers(0, 60, n), unit="D")).strftime("%Y-%m-%d"),
})
idx = rng.choice(n, 30, replace=False)
txns.loc[idx[:15], "amount"] = np.nan
txns.loc[idx[15:20], "amount"] = "N/A"
txns.loc[idx[20:25], "amount"] = "250000"
txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")
txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)
txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)
txns.to_csv("agent_transactions.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd

def clean_agent_data(txns: pd.DataFrame, agents: pd.DataFrame):
    """Clean, audit, and enrich agent transactions with the agent register.

    The returned log is a list of dictionaries with ``issue``, ``dimension``,
    ``column``, and ``rows_affected`` keys so it can be displayed as a table.
    Missing amounts are imputed with the median of valid amounts and retained
    in ``amount_imputed``; suspicious values are retained and flagged.
    """
    df = txns.copy()
    log = []

    def record(issue, dimension, column, rows_affected):
        log.append({
            "issue": issue,
            "dimension": dimension,
            "column": column,
            "rows_affected": int(rows_affected),
        })

    duplicate_count = df.duplicated("txn_id").sum()
    record("Duplicate transaction IDs", "Uniqueness", "txn_id", duplicate_count)
    df = df.drop_duplicates("txn_id", keep="first").reset_index(drop=True)

    raw_type = df["txn_type"].astype("string")
    normalized_type = raw_type.str.strip().str.lower().map({
        "deposit": "Deposit",
        "withdrawal": "Withdrawal",
    })
    formatting_type_count = (raw_type.notna() & ~raw_type.isin(["Deposit", "Withdrawal"])).sum()
    record("Non-standard transaction type formatting", "Consistency", "txn_type", formatting_type_count)
    invalid_type_count = normalized_type.isna().sum()
    record("Missing or invalid transaction types", "Consistency", "txn_type", invalid_type_count)
    df["txn_type"] = normalized_type
    if invalid_type_count:
        df = df.loc[df["txn_type"].notna()].copy()

    raw_amount = df["amount"].astype("string").replace({"N/A": np.nan, "": np.nan})
    numeric_amount = pd.to_numeric(raw_amount, errors="coerce")
    invalid_amount_count = numeric_amount.isna().sum()
    record("Missing or non-numeric amounts", "Completeness", "amount", invalid_amount_count)
    df["amount_imputed"] = numeric_amount.isna()
    median_amount = numeric_amount.median()
    df["amount"] = numeric_amount.fillna(median_amount)

    q1 = df["amount"].quantile(0.25)
    q3 = df["amount"].quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    df["is_outlier"] = (df["amount"] < lower_fence) | (df["amount"] > upper_fence)
    record("IQR outliers retained and flagged", "Validity", "amount", df["is_outlier"].sum())

    raw_dates = df["txn_date"].astype("string")
    non_iso_date_count = (~raw_dates.str.fullmatch(r"\d{4}-\d{2}-\d{2}")).sum()
    record("Non-ISO transaction date formatting", "Consistency", "txn_date", non_iso_date_count)
    parsed_dates = pd.to_datetime(raw_dates.str.replace("/", "-", regex=False), errors="coerce")
    invalid_date_count = parsed_dates.isna().sum()
    record("Invalid transaction dates", "Validity", "txn_date", invalid_date_count)
    df["txn_date"] = parsed_dates.dt.strftime("%Y-%m-%d")

    merged_df = df.merge(agents, on="agent_id", how="left", validate="many_to_one")
    unknown_agent_count = merged_df["county"].isna().sum()
    record("Unknown agent IDs", "Referential integrity", "agent_id", unknown_agent_count)

    return merged_df, log

## Executive Summary

- Duka Electronics generated KES 110,255,000 from 1,000 orders during January to June 2026.
- Nakuru led the branches with KES 24,326,000 in revenue and 207 orders.
- Laptops led product revenue at KES 52,815,000, while Smartwatches and Headphones each led units sold with 664 units.
- January was the strongest month at KES 20,238,000, compared with June at KES 16,777,000, a 20.6% difference.
- There were 82 Premium Online orders from Nairobi and Mombasa, worth KES 14,890,000 in total.

## First-five-minutes inspection

The first inspection checks the shape, column types, missing values, date range, and key categories before any transformation. This makes unexpected schema, completeness, time-window, or category problems visible early.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('duka_sales.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Run python generate_data.py from the repository root first.')

sales = pd.read_csv(DATA_PATH, parse_dates=['order_date'])
print('Shape:', sales.shape)
print('Types:')
print(sales.dtypes)
print('Missing values:')
print(sales.isna().sum())
print('Date range:', sales['order_date'].min().date(), 'to', sales['order_date'].max().date())
print('Branches:', sorted(sales['branch'].unique()))
print('Products:', sorted(sales['product'].unique()))
print('Channels:', sorted(sales['channel'].unique()))

Shape: (1000, 7)
Types:
order_id               int64
order_date    datetime64[ns]
branch                object
product               object
quantity               int64
channel               object
unit_price             int64
dtype: object
Missing values:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64
Date range: 2026-01-01 to 2026-06-30
Branches: ['Eldoret', 'Kisumu', 'Mombasa', 'Nairobi', 'Nakuru']
Products: ['Headphones', 'Laptop', 'Phone', 'Smartwatch', 'Tablet']
Channels: ['In-store', 'Online', 'Phone order']


## Derived columns

In [ ]:
sales['revenue'] = sales['quantity'] * sales['unit_price']
sales['month'] = sales['order_date'].dt.to_period('M').astype(str)
sales['weekday'] = sales['order_date'].dt.day_name()
sales['is_weekend'] = sales['order_date'].dt.dayofweek >= 5
sales['price_band'] = sales['unit_price'].ge(28000).map({True: 'Premium', False: 'Value'})
sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-01-01 00:00:00.000000000,Eldoret,Tablet,3,In-store,28000,84000,2026-01,Thursday,False,Premium
1,1002,2026-01-01 04:19:27.567567567,Nairobi,Smartwatch,5,Online,18000,90000,2026-01,Thursday,False,Value
2,1003,2026-01-01 08:38:55.135135135,Nairobi,Headphones,2,In-store,8500,17000,2026-01,Thursday,False,Value
3,1004,2026-01-01 12:58:22.702702702,Kisumu,Phone,1,Online,40000,40000,2026-01,Thursday,False,Premium
4,1005,2026-01-01 17:17:50.270270270,Mombasa,Tablet,4,Online,28000,112000,2026-01,Thursday,False,Premium


## Task 3: Branch rankings

Revenue and order rankings are not necessarily identical: revenue also reflects product mix and quantity, while order count gives every order equal weight.

In [ ]:
branch_summary = sales.groupby('branch').agg(
    total_revenue=('revenue', 'sum'),
    orders=('order_id', 'nunique')
).sort_values('total_revenue', ascending=False)
branch_summary

,total_revenue,orders
branch,,
Nakuru,24326000,207
Mombasa,22796000,212
Nairobi,22228500,197
Eldoret,21926000,194
Kisumu,18978500,190


## Task 4: Product performance

The revenue and units lists can differ because products have different unit prices. High-priced products can lead revenue even when lower-priced products sell more units.

In [ ]:
product_summary = sales.groupby('product').agg(
    revenue=('revenue', 'sum'),
    units_sold=('quantity', 'sum')
).sort_values('revenue', ascending=False)
product_summary

,revenue,units_sold
product,,
Laptop,52815000,503
Phone,23800000,595
Tablet,16044000,573
Smartwatch,11952000,664
Headphones,5644000,664


## Task 5: Weekend versus weekday orders

This comparison uses order count, total revenue, and average revenue per order so volume and value are considered together.

In [ ]:
weekend_summary = sales.groupby(sales['is_weekend'].map({True: 'Weekend', False: 'Weekday'})).agg(
    orders=('order_id', 'nunique'),
    total_revenue=('revenue', 'sum'),
    average_order_revenue=('revenue', 'mean')
)
weekend_summary

,orders,total_revenue,average_order_revenue
is_weekend,,,
Weekday,711,79995000,112510.548523
Weekend,289,30260000,104705.882353


## Task 6: Best and worst month

The percentage difference is calculated as `(best month - worst month) / worst month * 100`.

In [ ]:
monthly_revenue = sales.groupby('month')['revenue'].sum().sort_index()
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()
percentage_difference = (monthly_revenue[best_month] - monthly_revenue[worst_month]) / monthly_revenue[worst_month] * 100
print('Best:', best_month, monthly_revenue[best_month])
print('Worst:', worst_month, monthly_revenue[worst_month])
print('Percentage difference:', round(percentage_difference, 1), '%')
monthly_revenue

Best: 2026-01 20238000
Worst: 2026-06 16777000
Percentage difference: 20.6 %


,revenue
month,
2026-01,20238000
2026-02,17494000
2026-03,18289000
2026-04,18347000
2026-05,19110000
2026-06,16777000


## Task 7: Premium online coastal-capital orders

The requested filter keeps Premium orders sold Online by the Nairobi or Mombasa branches and exports the complete filtered rows without a pandas index.

In [ ]:
from pathlib import Path

premium_online_coast_capital = sales.loc[
    sales['price_band'].eq('Premium')
    & sales['channel'].eq('Online')
    & sales['branch'].isin(['Nairobi', 'Mombasa'])
].copy()

# Ensure the outputs directory exists
Path('outputs').mkdir(parents=True, exist_ok=True)

premium_online_coast_capital.to_csv('outputs/premium_online_coast_capital.csv', index=False)
print('Orders:', len(premium_online_coast_capital))
print('Total revenue:', premium_online_coast_capital['revenue'].sum())
premium_online_coast_capital.head()

Orders: 82
Total revenue: 14890000


,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
4,1005,2026-01-01 17:17:50.270270270,Mombasa,Tablet,4,Online,28000,112000,2026-01,Thursday,False,Premium
24,1025,2026-01-05 07:47:01.621621621,Mombasa,Laptop,1,Online,105000,105000,2026-01,Monday,False,Premium
38,1039,2026-01-07 20:19:27.567567567,Nairobi,Tablet,5,Online,28000,140000,2026-01,Wednesday,False,Premium
68,1069,2026-01-13 06:03:14.594594594,Mombasa,Laptop,5,Online,105000,525000,2026-01,Tuesday,False,Premium
86,1087,2026-01-16 11:53:30.810810811,Nairobi,Tablet,4,Online,28000,112000,2026-01,Friday,False,Premium


## Task 8: Spreadsheet reproduction

The workbook contains a SUMIFS branch revenue table, an XLOOKUP region column, and a PivotTable-style revenue summary by branch and channel. The numeric totals are checked against the pandas result before saving.

In [ ]:
from pathlib import Path

region_map = pd.DataFrame({
    'branch': ['Nairobi', 'Mombasa', 'Nakuru', 'Kisumu', 'Eldoret'],
    'region': ['Capital', 'Coast', 'Rift Valley', 'Nyanza', 'Rift Valley']
})
branch_order = sales.groupby('branch')['revenue'].sum().sort_values(ascending=False).index.tolist()
branch_table = pd.DataFrame({'branch': branch_order})
branch_table['pandas_revenue'] = branch_table['branch'].map(sales.groupby('branch')['revenue'].sum())
branch_table['orders'] = branch_table['branch'].map(sales.groupby('branch')['order_id'].nunique())
branch_table['SUMIFS_revenue'] = [f'=SUMIFS(SalesData!$G:$G,SalesData!$C:$C,A{row})' for row in range(3, 3 + len(branch_table))]
branch_table['region_XLOOKUP'] = [f'=XLOOKUP(A{row},Regions!$A$2:$A$6,Regions!$B$2:$B$6,"Unknown")' for row in range(3, 3 + len(branch_table))]
pivot_table = sales.pivot_table(index='branch', columns='channel', values='revenue', aggfunc='sum', fill_value=0, margins=True, margins_name='Grand Total')
Path('excel').mkdir(exist_ok=True)
with pd.ExcelWriter('excel/duka_branch_summary.xlsx') as writer:
    sales.to_excel(writer, sheet_name='SalesData', index=False)
    region_map.to_excel(writer, sheet_name='Regions', index=False)
    branch_table.to_excel(writer, sheet_name='BranchSummary', index=False, startrow=1)
    pivot_table.to_excel(writer, sheet_name='PivotTable')

print('Pandas total:', sales['revenue'].sum())
print('Pivot total:', pivot_table.loc['Grand Total', 'Grand Total'])
print('Totals match:', sales['revenue'].sum() == pivot_table.loc['Grand Total', 'Grand Total'])

Pandas total: 110255000
Pivot total: 110255000
Totals match: True


## Task 9: Executive Summary

The five numbered findings at the top are the executive summary: they give leadership the overall revenue, leading branch, product results, monthly comparison, and requested filtered-order result with specific numbers.

In [ ]:
import pandas as pd

txns = pd.read_csv('agent_transactions.csv', keep_default_na=True)
agents = pd.read_csv('agents.csv')
print('Raw transaction rows:', len(txns))
print('Agent rows:', len(agents))
print('Raw types:', sorted(txns['txn_type'].dropna().unique()))
print('Raw amount missing:', txns['amount'].isna().sum())
clean_df, cleaning_log = clean_agent_data(txns, agents)
audit_table = pd.DataFrame(cleaning_log, columns=['issue', 'dimension', 'column', 'rows_affected'])
audit_table

Raw transaction rows: 612
Agent rows: 20
Raw types: ['DEPOSIT ', 'Deposit', 'Withdrawal', 'deposit', 'withdrawal']
Raw amount missing: 22


,issue,dimension,column,rows_affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Non-standard transaction type formatting,Consistency,txn_type,376
2,Missing or invalid transaction types,Consistency,txn_type,0
3,Missing or non-numeric amounts,Completeness,amount,20
4,IQR outliers retained and flagged,Validity,amount,53
5,Non-ISO transaction date formatting,Consistency,txn_date,5
6,Invalid transaction dates,Validity,txn_date,0
7,Unknown agent IDs,Referential integrity,agent_id,7


## Cleaning decisions

Duplicate transaction IDs are reduced to one record per ID. Transaction types are stripped, lowercased, mapped to exactly `Deposit` or `Withdrawal`, and invalid or missing types would be excluded. Amounts are coerced to numeric; missing or invalid values are imputed with the median of valid amounts and marked in `amount_imputed`, preserving rows while making the treatment visible. Dates accept both hyphen and slash separators and are saved as ISO dates. IQR outliers are retained and flagged because an extreme value may be a genuine transaction requiring business investigation rather than automatic deletion. Unknown agents are retained after a left merge with a missing county so the governance issue remains countable.

## Post-merge validation

In [ ]:
validation = pd.DataFrame({
    'measure': ['Raw transaction rows', 'Clean transaction rows', 'Unique txn_id values', 'Unknown-agent transactions', 'Duplicate txn_id values after cleaning'],
    'value': [len(txns), len(clean_df), clean_df['txn_id'].nunique(), clean_df['county'].isna().sum(), clean_df['txn_id'].duplicated().sum()]
})
validation

,measure,value
0,Raw transaction rows,612
1,Clean transaction rows,600
2,Unique txn_id values,600
3,Unknown-agent transactions,7
4,Duplicate txn_id values after cleaning,0


## County summary

In [ ]:
county_summary = clean_df.groupby(['county', 'txn_type'], dropna=False).agg(
    total_value=('amount', 'sum'),
    number_of_transactions=('txn_id', 'nunique'),
    average_value=('amount', 'mean')
).reset_index()
county_summary

,county,txn_type,total_value,number_of_transactions,average_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77050.5,27,2853.722222
2,Kisumu,Deposit,436420.5,74,5897.574324
3,Kisumu,Withdrawal,76374.5,37,2064.175676
4,Mombasa,Deposit,148986.0,72,2069.25
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543590.0,118,4606.694915
7,Nairobi,Withdrawal,162036.0,62,2613.483871
8,Nakuru,Deposit,184890.0,60,3081.5
9,Nakuru,Withdrawal,328594.5,39,8425.5


## Weekly transaction-value pivot

In [ ]:
clean_df['week'] = pd.to_datetime(clean_df['txn_date']).dt.to_period('W').astype(str)
weekly_pivot = clean_df.pivot_table(
    index='week', columns='county', values='amount', aggfunc='sum', fill_value=0, dropna=False
)
weekly_pivot

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,NaN
week,,,,,,
2026-02-23/2026-03-01,0.0,1893.0,6759.0,7046.0,2743.5,0.0
2026-03-02/2026-03-08,18317.0,287933.0,285669.5,68967.0,56574.0,0.0
2026-03-09/2026-03-15,267771.5,23186.0,30477.5,52860.5,28730.0,0.0
2026-03-16/2026-03-22,22054.0,28125.0,25144.5,43107.5,39795.0,1639.0
2026-03-23/2026-03-29,11504.0,27077.0,17050.0,41373.5,18998.5,0.0
2026-03-30/2026-04-05,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0
2026-04-06/2026-04-12,15030.0,15341.0,47092.5,56786.5,276609.0,9824.0
2026-04-13/2026-04-19,12265.0,29416.0,26730.0,56432.5,26152.5,0.0
2026-04-20/2026-04-26,24507.0,37439.0,16687.0,65187.0,30862.0,1877.0


In [ ]:
from pathlib import Path

# Ensure the outputs directory exists before saving
Path('outputs').mkdir(parents=True, exist_ok=True)

clean_df.to_csv('outputs/agent_transactions_clean.csv', index=False)
print('Exported rows:', len(clean_df))
print('txn_type values:', sorted(clean_df['txn_type'].unique()))
print('ISO dates:', clean_df['txn_date'].str.fullmatch(r'\d{4}-\d{2}-\d{2}').all())

Exported rows: 600
txn_type values: ['Deposit', 'Withdrawal']
ISO dates: True


# Data Quality Report

## Uniqueness
The transaction extract contains repeated `txn_id` values. The pipeline keeps the first record for each ID, logs the number removed, and verifies that the cleaned output has no duplicate IDs.

## Consistency
Transaction-type spelling and case variants are standardised to exactly `Deposit` and `Withdrawal`. Missing or unrecognised values are identified before mapping and would be excluded rather than silently converted.

## Validity
Amounts are converted to numeric values and dates are parsed after normalising slash separators, then written in ISO `YYYY-MM-DD` form. IQR fences identify suspicious amounts; the KES 250,000 records are retained and flagged for operational investigation because deleting potentially genuine transactions would lose value.

## Completeness
Missing and non-numeric amounts are imputed with the median of valid amounts so valid transaction rows are not discarded. The `amount_imputed` flag preserves a transparent record of every replacement.

## Accuracy
The county and weekly summaries use the cleaned amount field, while outlier flags allow managers to compare reported totals with and without suspicious transactions. The pipeline is deterministic and records every treatment in its audit log.

## Referential integrity
A left join preserves every cleaned transaction and adds the agent county where the ID exists in the register. Unknown IDs remain visible with a missing county and are counted in post-merge validation rather than being silently dropped.

## Privacy
If customer phone numbers were present, they would be personal data under the Kenya Data Protection Act 2019. The business should collect them only for a defined lawful purpose, minimise access, encrypt them in transit and at rest, mask or pseudonymise them for analysis, restrict exports, and delete them when the retention purpose ends.